# AI Project Planning & Risk Forecasting App (Clean Capstone Notebook)

This notebook is the clean, step-by-step version of your final project.

Use it for:
- Implementation walkthrough
- Live demo rehearsal
- Stakeholder explanation and Q&A preparation

## What you will demonstrate
1. AI task generation (Groq / mock)
2. Workflow graph and critical path
3. Monte Carlo forecasting with uncertainty
4. Delay risk metrics (P50, P80, delay probability)
5. Delay driver ranking
6. Scenario decision support
7. SQLite session persistence

## Live presentation flow (recommended)
- Start with `APP_MODE=mock` for reliability
- Show one full run end-to-end
- Switch to `APP_MODE=real` only if API is stable
        

## 0. Environment Setup

Run this once in your project environment.

- Python 3.11+
- `requirements.txt` for app runtime
- `.env` for secrets/config
        

In [ ]:
# Uncomment if dependencies are not installed yet:
# !pip install -r requirements.txt

# Optional for tests/notebook extras:
# !pip install -r requirements-dev.txt

print("Environment setup step ready.")
        

## 1. Imports and Project Modules

This project is modular. We import from `src/` so notebook and Streamlit app share the same logic.
        

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure project root is importable if notebook is opened from a subfolder
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

from src.config import settings
from src.ai.schema import TaskPlan
from src.ai.task_generator import generate_task_plan
from src.modeling.graph_builder import build_project_graph
from src.modeling.critical_path import critical_path_by_mean
from src.simulation.monte_carlo import run_monte_carlo
from src.analytics.metrics import compute_metrics
from src.analytics.risk_drivers import rank_delay_drivers
from src.analytics.scenarios import scenario_comparison
from src.visualization.charts import (
    completion_histogram,
    dependency_graph_figure,
    risk_drivers_bar,
    scenario_comparison_chart,
)
from src.storage.db import init_db
from src.storage.repository import save_session_run, list_recent_runs, get_run_details

load_dotenv()
print("Imports loaded successfully.")
        

## 2. Configuration Check

### Explain this during demo
- `APP_MODE=mock`: no API calls, safe for rehearsal/live fallback
- `APP_MODE=real`: calls Groq API
- `GROQ_MODEL`: active model name
- `DEFAULT_ITERATIONS`: default simulation volume
        

In [ ]:
print("APP_MODE           :", settings.app_mode)
print("GROQ_MODEL         :", settings.groq_model)
print("DEFAULT_ITERATIONS :", settings.default_iterations)
print("MAX_ITERATIONS     :", settings.max_iterations)
print("MAX_TASKS          :", settings.max_tasks)
print("SQLITE_DB_PATH     :", settings.sqlite_db_path)
print("GROQ_API_KEY set?  :", "yes" if bool(settings.groq_api_key) else "no")
        

## 3. Define a Demo Scenario

Choose one realistic project statement. Keep it concrete.
        

In [ ]:
PROJECT_DESCRIPTION = "Build a mobile app MVP with login, payments, and analytics"
DEADLINE_DAYS = 90.0
ITERATIONS = min(1000, settings.max_iterations)
MAX_TASKS = min(12, settings.max_tasks)
SEED = 7
MODE = "mock"  # change to "real" for Groq API generation

print("Project:", PROJECT_DESCRIPTION)
print("Mode:", MODE)
print("Deadline (days):", DEADLINE_DAYS)
print("Iterations:", ITERATIONS)
print("Max tasks:", MAX_TASKS)
        

## 4. Generate and Validate Task Plan (AI Layer)

### Explain this during demo
- Prompted LLM returns JSON task plan
- Pydantic validates structure, IDs, dependencies, and risk ranges
- Invalid outputs are rejected with clear errors
        

In [ ]:
plan = generate_task_plan(
    project_description=PROJECT_DESCRIPTION,
    max_tasks=MAX_TASKS,
    mode=MODE,
)

# Strict schema check (already done in generate_task_plan, shown here for teaching)
TaskPlan.model_validate(plan.model_dump())

tasks = [task.model_dump() for task in plan.tasks]
df_tasks = pd.DataFrame(tasks)

print(f"Generated {len(tasks)} tasks.")
df_tasks
        

## 5. Build Workflow DAG and Critical Path

### Explain this during demo
- Tasks are nodes
- Dependencies are directed edges
- DAG validation prevents cycles
- Critical path gives baseline schedule pressure
        

In [ ]:
G = build_project_graph(tasks)
critical_path, critical_path_days = critical_path_by_mean(G)

print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
print("Critical path:", " -> ".join(critical_path))
print(f"Critical path baseline duration: {critical_path_days:.1f} days")
        

In [ ]:
fig_graph = dependency_graph_figure(G, critical_path)
fig_graph.show()
        

## 6. Monte Carlo Simulation (Dynamic Critical Path)

### Key technical point
In this implementation, the critical path is recomputed for each simulation run using sampled durations.

That makes the forecast more realistic than using one fixed path for all runs.
        

In [ ]:
completion_times, sampled_paths = run_monte_carlo(
    G,
    iterations=ITERATIONS,
    seed=SEED,
    return_paths=True,
)

unique_paths = {tuple(path) for path in sampled_paths}

print(f"Simulations: {len(completion_times)}")
print(f"Completion min/mean/max: {completion_times.min():.1f} / {completion_times.mean():.1f} / {completion_times.max():.1f}")
print(f"Unique sampled critical paths: {len(unique_paths)}")
        

## 7. Forecast Metrics and Executive Summary

Metrics to explain clearly:
- `P50`: median completion date
- `P80`: safer commitment date
- `Delay probability`: probability completion exceeds deadline
        

In [ ]:
metrics = compute_metrics(completion_times, deadline_days=DEADLINE_DAYS)


def risk_level(delay_probability: float) -> str:
    if delay_probability < 0.25:
        return "LOW"
    if delay_probability < 0.55:
        return "MEDIUM"
    return "HIGH"

level = risk_level(metrics["delay_probability"])

print("Mean completion:", round(metrics["mean"], 1), "days")
print("P50:", round(metrics["p50"], 1), "days")
print("P80:", round(metrics["p80"], 1), "days")
print("Delay probability:", round(metrics["delay_probability"] * 100, 1), "%")
print("Risk level:", level)

summary = (
    f"Forecast: {metrics['delay_probability']*100:.1f}% probability of missing the "
    f"{DEADLINE_DAYS:.0f}-day deadline (risk: {level}). "
    f"Recommended commitment date: {metrics['p80']:.1f} days (P80)."
)
print("
Executive Summary:")
print(summary)
        

In [ ]:
fig_hist = completion_histogram(
    completion_times,
    deadline=DEADLINE_DAYS,
    p50=metrics["p50"],
    p80=metrics["p80"],
)
fig_hist.show()
        

## 8. Delay Driver Ranking

This identifies which tasks most frequently appear on sampled critical paths.

Use this to justify mitigation priorities to stakeholders.
        

In [ ]:
drivers = rank_delay_drivers(G, iterations=min(500, ITERATIONS), seed=SEED)
df_drivers = pd.DataFrame(drivers)

print("Top delay drivers:")
df_drivers.head(10)
        

In [ ]:
if not df_drivers.empty:
    fig_drivers = risk_drivers_bar(df_drivers.head(10))
    fig_drivers.show()
        

## 9. Scenario Decision Support

Three stakeholder-relevant scenarios:
1. Baseline
2. Aggressive deadline (-15%)
3. Increased capacity (+15% faster execution)
        

In [ ]:
df_scenarios = scenario_comparison(
    build_graph_fn=build_project_graph,
    tasks=tasks,
    deadline_days=DEADLINE_DAYS,
    iterations=ITERATIONS,
    seed=SEED,
)

df_scenarios
        

In [ ]:
fig_scenarios = scenario_comparison_chart(df_scenarios)
fig_scenarios.show()
        

## 10. SQLite Persistence (Session History)

### Why this matters
- Enables auditability
- Enables stakeholder follow-up
- Lets you reload runs during demos
        

In [ ]:
init_db()

run_id = save_session_run(
    project_text=PROJECT_DESCRIPTION,
    mode=MODE,
    params={
        "max_tasks": MAX_TASKS,
        "iterations": ITERATIONS,
        "seed": SEED,
    },
    tasks=tasks,
    deadline_days=DEADLINE_DAYS,
    iterations=ITERATIONS,
    seed=SEED,
    metrics=metrics,
    completion_times=completion_times,
    scenarios_df=df_scenarios,
)

print("Saved simulation_run_id:", run_id)
        

In [ ]:
recent = list_recent_runs(limit=5)
pd.DataFrame(recent)
        

In [ ]:
details = get_run_details(run_id)
print("Loaded run details keys:")
print(sorted(details.keys()) if details else "No details found")
        

## 11. Export Artifacts for Presentation

These exports are useful during evaluation:
- `task_plan.csv`
- `scenario_comparison.csv`
- `risk_drivers.csv`
- `run_summary.json`
        

In [ ]:
out_dir = Path("exports")
out_dir.mkdir(exist_ok=True)

pd.DataFrame(tasks).to_csv(out_dir / "task_plan.csv", index=False)
df_scenarios.to_csv(out_dir / "scenario_comparison.csv", index=False)
df_drivers.to_csv(out_dir / "risk_drivers.csv", index=False)

summary_payload = {
    "project_description": PROJECT_DESCRIPTION,
    "mode": MODE,
    "deadline_days": DEADLINE_DAYS,
    "iterations": ITERATIONS,
    "critical_path": critical_path,
    "critical_path_days": critical_path_days,
    "metrics": metrics,
    "top_drivers": drivers[:5],
    "scenarios": df_scenarios.to_dict(orient="records"),
}

(out_dir / "run_summary.json").write_text(json.dumps(summary_payload, indent=2))
print("Exported files to:", out_dir.resolve())
        

## 12. Run the Full UI App

Use this outside the notebook in terminal:

```bash
streamlit run app.py
```

Optional Docker run:

```bash
docker compose up --build
```
        

## 13. Stakeholder Q&A Prep (Use in Presentation)

### Q1: Why P80 instead of P50 for commitment?
P50 is median, but P80 includes safety against uncertainty, making commitments more reliable.

### Q2: Why Monte Carlo?
It models uncertainty in durations, producing probability distributions instead of false single-point certainty.

### Q3: How do you identify where to intervene?
Risk driver ranking shows which tasks most often dominate the critical path.

### Q4: Why scenario comparison?
Stakeholders need decisions, not only forecasts. Scenarios convert analytics into action options.

### Q5: Why SQLite?
It is lightweight, local, easy for capstone demos, and enough for session persistence.

### Q6: Why mock mode?
It guarantees demo continuity if API latency/outage occurs.
        

## 14. Team Implementation Plan (2 People, 4 Weeks)

### Person A (Product + Presentation)
- Owns UI flow, stakeholder narrative, and demo quality
- Validates business framing and interpretation of scenarios

### Person B (Engineering + Analytics)
- Owns simulation correctness, persistence, and testing
- Validates technical robustness and reproducibility

### Joint deliverables
- One live deployment (Streamlit Cloud)
- One Docker-based local reproducible run
- One final deck + live demo + backup demo path
        

## 15. Definition of Done (Final Checklist)

- [ ] `streamlit run app.py` works locally
- [ ] Mock mode demo is stable end-to-end
- [ ] Real mode works with valid `GROQ_API_KEY`
- [ ] Scenario table and driver ranking are explained clearly
- [ ] Exports are generated
- [ ] SQLite history loads previous runs
- [ ] Presentation rehearsal completed by both team members
        